<a href="https://colab.research.google.com/github/jorgemunozl/vla-test/blob/main/sixth_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test For Orbit

Difference with FiS and related

In [1]:
import os

# Test
# Set this BEFORE importing pytorch/tensorflow
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
# Check if it worked (should return 1 if you selected a single GPU)
print(torch.cuda.device_count()) 

1


In [2]:
# DS_ID = "NONHUMAN-RESEARCH/general-task-index"
DS_ID = "NONHUMAN-RESEARCH/pick-and-place-fruits_v2_cleaned"
PRETRAINED_PATH = "lerobot/pi05_base"

In [3]:
from xhuman.policies.orbit.configuration_orbit import OrbitConfig

policy_config = OrbitConfig(repo_id="none",device="cuda",pretrained_path=PRETRAINED_PATH)

In [4]:
from xhuman.configs.default import LerobotDatasetConfig

dataset_config = LerobotDatasetConfig(
    repo_id=DS_ID,
)

In [5]:
from xhuman.configs.train import TrainPipelineConfigXHUMAN

cfg = TrainPipelineConfigXHUMAN(
    dataset=dataset_config,
    policy=policy_config,
)
cfg.validate()

In [6]:
from xhuman.datasets.factory import make_dataset_xhuman

dataset = make_dataset_xhuman(cfg)

Fetching 59 files:   0%|          | 0/59 [00:00<?, ?it/s]

Fetching 59 files:   0%|          | 0/59 [00:00<?, ?it/s]

In [7]:
policy_config.validate_features()

In [8]:
from xhuman.policies.factory import make_xhuman_pre_post_processors

preprocessor, _ = make_xhuman_pre_post_processors(
        policy_cfg=policy_config,
        dataset_stats=dataset.meta.stats,
    )

In [9]:
for i in range(6):
    print(preprocessor[i])

RenameObservationsProcessorStep(rename_map={})
AddBatchDimensionProcessorStep(to_batch_action_processor=AddBatchDimensionActionStep(), to_batch_observation_processor=AddBatchDimensionObservationStep(), to_batch_complementary_data_processor=AddBatchDimensionComplementaryDataStep())
RelativeActionsProcessorStep(enabled=False, exclude_joints=['gripper'], action_names=None)
NormalizerProcessorStep(features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,)), 'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, norm_map={<FeatureType.VISUAL: 'VISUAL'>: <NormalizationMode.IDENTITY: 'IDENTITY'>, <FeatureType.STATE: 'STATE'>: <NormalizationMode.QUANTILES: 'QUANTILES'>, <FeatureType.ACTION: 'ACTION'>: <NormalizationMode.QUANTILES: 'QUANTILES'>}, stats={'observation.images.right': {'min': array([[[0.]],

       [[0.]],

       [[0.]]]), 'max': array([[[1.]],

       [[1.]],

       [[1.]]]), 'mean': tensor([[[0.4850]],

        [[0.4560]],



In [10]:
preprocessor

DataProcessorPipeline(name='policy_preprocessor', steps=7: [RenameObservationsProcessorStep, AddBatchDimensionProcessorStep, ..., DeviceProcessorStep])

In [11]:
device = torch.device("cuda")

In [12]:
frame = dataset[0]
frame.keys()

dict_keys(['observation.images.left', 'observation.images.top', 'observation.images.right', 'action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'action_is_pad', 'task'])

In [13]:
frame["observation.state"]

tensor([ -4.0033, -99.8244,  99.8024,  -3.4150,  31.1800,  12.3139,   0.0000,
         -1.5907, -95.7211,  99.7129,  -1.0590,  37.7243,  11.8261,   0.8400])

In [14]:
processed = preprocessor(frame)

In [15]:
processed.keys()

dict_keys(['action', 'next.reward', 'next.done', 'next.truncated', 'info', 'action_is_pad', 'task', 'index', 'task_index', 'episode_index', 'observation.images.left', 'observation.images.top', 'observation.images.right', 'observation.state', 'observation.language.tokens', 'observation.language.attention_mask'])

In [16]:
processed["observation.state"]

tensor([[-1.2679, -0.9978,  1.0026, -0.2297, -0.9334,  0.6216, -0.9995,  0.3998,
         -0.9995,  0.9993, -0.1850, -0.6628,  0.2763, -0.9757]],
       device='cuda:0')

In [17]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")

In [18]:
tokens_pt = batch_slow["observation.language.tokens"]
words = tokenizer.batch_decode(tokens_pt, skip_special_tokens=True)
words

NameError: name 'batch_slow' is not defined

In [ ]:
prefix_out , kv_cache , pad_mask  = policy.sample_embedding(batch_slow)

In [ ]:
prefix_out[0].shape

In [ ]:
batch_slow.keys()

In [ ]:
img_fast, img_masks_fast = policy._preprocess_images(batch_fast)
raw_state = batch_fast["observation.state"]
raw_state

In [ ]:
batch_slow["observation.state"]

In [ ]:
policy.model.config.output_features["action"].shape

In [ ]:
actions = policy.model.sample_actions(
    images=img_fast,
    img_masks=img_masks_fast,
    tokens=tokens_slow,
    masks=masks_slow,
)
actions.shape

In [ ]:
actions = policy.model.fast_model(
    images_fast=img_fast,
    img_masks_fast=img_masks_fast,
    past_key_values=kv_cache,
    state_embs=raw_state,
    latent_embeds=prefix_out[0],
    prefix_pad_masks=pad_mask,
)
actions.shape